# FLOWCRED AI
## Late-Payment Feature Engineering

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from feature_engine.datetime import DatetimeFeatures
from feature_engine.transformation import LogTransformer
from feature_engineering_utils import LeakageSafeCustomerHistoryTransformer, build_feature_pipeline

repo_root = Path(r"D:/ML_CP")
clean_path = repo_root / "data" / "processed" / "flowcred_cleaned.csv"
feature_output_path = repo_root / "data" / "processed" / "late_payment_features.csv"
train_output_path = repo_root / "data" / "processed" / "late_payment_train.csv"
validation_output_path = repo_root / "data" / "processed" / "late_payment_validation.csv"
test_output_path = repo_root / "data" / "processed" / "late_payment_test.csv"
report_dir = repo_root / "reports"
report_path = report_dir / "late_payment_feature_dictionary.csv"
pipeline_output_path = repo_root / "models" / "late_payment_feature_pipeline.joblib"

print("repo_root:", repo_root)
print("clean_path:", clean_path)
print("feature_output_path:", feature_output_path)
print("pipeline_output_path:", pipeline_output_path)


## 1. Load Data

In [ ]:
df = pd.read_csv(clean_path)
required = [
    "invoice_id",
    "customer_id",
    "invoice_amount",
    "invoice_date",
    "due_date",
    "payment_date",
    "delay_days",
    "is_late",
    "invoice_status",
]
missing = [col for col in required if col not in df.columns]
print("Missing columns:", missing)
assert not missing, f"Required columns missing: {missing}"
print("Dataset shape:", df.shape)


## 2. Verify Dataset

In [ ]:
for col in ["invoice_amount", "delay_days", "is_late"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
for col in ["invoice_date", "due_date", "payment_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
print(df.dtypes)
print("\nMissing value summary:")
display(df.isna().sum().sort_values(ascending=False).head(10))


## 3. Select Paid Invoices

In [ ]:
paid_df = df[df["payment_date"].notna()].copy()
print("Paid invoices:", len(paid_df))
print("Missing payment_date after filter:", int(paid_df["payment_date"].isna().sum()))
print("Missing delay_days after filter:", int(paid_df["delay_days"].isna().sum()))
print("Missing is_late after filter:", int(paid_df["is_late"].isna().sum()))
assert paid_df["payment_date"].notna().all()
assert paid_df["delay_days"].notna().all()
assert paid_df["is_late"].notna().all()


## 4. Verify Target

In [ ]:
paid_df = paid_df.sort_values(["invoice_date", "invoice_id"], ascending=[True, True]).reset_index(drop=True)
expected_is_late = (paid_df["payment_date"] > paid_df["due_date"]).astype(int)
mismatch_mask = paid_df["is_late"].fillna(-1).astype(float) != expected_is_late.astype(float)
print("Target mismatch count:", int(mismatch_mask.sum()))
if mismatch_mask.any():
    display(paid_df.loc[mismatch_mask, ["invoice_id", "customer_id", "invoice_date", "due_date", "payment_date", "is_late"]].head(10))
else:
    print("No target mismatches found: existing is_late matches payment_date > due_date.")
assert not mismatch_mask.any(), "Target verification failed."


## 5. Chronological Ordering

In [ ]:
paid_df = paid_df.sort_values(["invoice_date", "invoice_id"], ascending=[True, True]).reset_index(drop=True)
print(paid_df[["invoice_id", "customer_id", "invoice_date", "due_date", "payment_date", "is_late"]].head(10).to_string(index=False))


## 6. Feature-engine Based Feature Engineering

In [ ]:
datetime_transformer = DatetimeFeatures(
    variables=["invoice_date"],
    features_to_extract=["year", "month", "quarter", "day_of_week", "day_of_month", "week", "weekend"],
    drop_original=True,
)

invoice_dt = datetime_transformer.fit_transform(paid_df[["invoice_date"]].copy())
invoice_dt.columns = [
    "invoice_year",
    "invoice_month",
    "invoice_quarter",
    "invoice_day_of_week",
    "invoice_day_of_month",
    "invoice_week_of_year",
    "invoice_is_weekend",
]
print("Datetime features preview:")
display(invoice_dt.head())

log_transformer = LogTransformer(variables=["invoice_amount"])
log_amount = log_transformer.fit_transform(paid_df[["invoice_amount"]].copy())
log_amount.columns = ["log_invoice_amount"]
print("\nLog invoice amount preview:")
display(log_amount.head())

paid_df["days_until_due"] = (paid_df["due_date"] - paid_df["invoice_date"]).dt.days
print("\nSample days_until_due:")
display(paid_df[["invoice_id", "invoice_date", "due_date", "days_until_due"]].head())


## 7. Custom Leakage-Safe Customer History Features

In [ ]:
history_transformer = LeakageSafeCustomerHistoryTransformer(
    customer_col="customer_id",
    date_col="invoice_date",
    target_col="is_late",
    amount_col="invoice_amount",
    delay_col="delay_days",
    invoice_id_col="invoice_id",
)
customer_history = history_transformer.fit_transform(paid_df)
print("Customer history features preview:")
display(customer_history.head(10))


## 8. Feature Validation

In [ ]:
feature_frame = pd.concat([
    paid_df[["invoice_id", "customer_id", "invoice_date", "is_late", "invoice_amount", "days_until_due"]].reset_index(drop=True),
    invoice_dt.reset_index(drop=True),
    log_amount.reset_index(drop=True),
    customer_history.reset_index(drop=True),
], axis=1)
print("Feature frame shape:", feature_frame.shape)
print("Feature frame columns:", feature_frame.columns.tolist()[:15])

assert feature_frame["invoice_id"].is_unique
assert len(feature_frame) == len(paid_df)


## 9. Chronological Train/Validation/Test Split

In [ ]:
feature_frame = feature_frame.sort_values(["invoice_date", "invoice_id"], ascending=[True, True]).reset_index(drop=True)
train_end = int(len(feature_frame) * 0.70)
val_end = int(len(feature_frame) * 0.85)
train = feature_frame.iloc[:train_end].copy()
validation = feature_frame.iloc[train_end:val_end].copy()
test = feature_frame.iloc[val_end:].copy()
print("Train rows:", len(train))
print("Validation rows:", len(validation))
print("Test rows:", len(test))
print("Train max date:", train["invoice_date"].max())
print("Validation min date:", validation["invoice_date"].min())
print("Validation max date:", validation["invoice_date"].max())
print("Test min date:", test["invoice_date"].min())
assert train["invoice_date"].max() <= validation["invoice_date"].min()
assert validation["invoice_date"].max() <= test["invoice_date"].min()
print("Temporal split integrity passed.")


## 10. Feature Preprocessing Pipeline

In [ ]:
final_feature_cols = [
    "invoice_amount",
    "log_invoice_amount",
    "invoice_year",
    "invoice_month",
    "invoice_quarter",
    "invoice_day_of_week",
    "invoice_day_of_month",
    "invoice_week_of_year",
    "invoice_is_weekend",
    "days_until_due",
    "previous_invoice_count",
    "previous_late_count",
    "previous_late_ratio",
    "previous_avg_delay",
    "previous_median_delay",
    "previous_avg_invoice_amount",
    "previous_total_invoice_amount",
    "days_since_previous_invoice",
    "has_payment_history",
]
missing_features = [col for col in final_feature_cols if col not in feature_frame.columns]
assert not missing_features, f"Expected engineered features missing: {missing_features}"
print("Final feature columns:")
print(final_feature_cols)
feature_pipeline = build_feature_pipeline(final_feature_cols)
feature_pipeline.fit(train[final_feature_cols])
train_trans = feature_pipeline.transform(train[final_feature_cols])
validation_trans = feature_pipeline.transform(validation[final_feature_cols])
test_trans = feature_pipeline.transform(test[final_feature_cols])
print("Train transformed shape:", train_trans.shape)
print("Validation transformed shape:", validation_trans.shape)
print("Test transformed shape:", test_trans.shape)


## 11. Leakage Audit

In [ ]:
FORBIDDEN_FEATURES = {"invoice_id", "customer_id", "payment_date", "delay_days", "is_late", "invoice_status"}
assert len(final_feature_cols) == 19, f"Expected 19 model features, got {len(final_feature_cols)}"
assert set(final_feature_cols).isdisjoint(FORBIDDEN_FEATURES), "Forbidden leakage features present."
assert feature_frame["invoice_id"].is_unique, "Duplicate invoices in feature frame."
assert set(train["invoice_id"]).isdisjoint(set(validation["invoice_id"]))
assert set(train["invoice_id"]).isdisjoint(set(test["invoice_id"]))
assert set(validation["invoice_id"]).isdisjoint(set(test["invoice_id"]))
assert len(train) + len(validation) + len(test) == len(feature_frame)
assert train["invoice_date"].max() <= validation["invoice_date"].min()
assert validation["invoice_date"].max() <= test["invoice_date"].min()

history_features = [
    "previous_late_count", "previous_late_ratio", "previous_avg_delay",
    "previous_median_delay", "previous_avg_invoice_amount",
    "previous_total_invoice_amount", "days_since_previous_invoice",
]
assert feature_frame.loc[feature_frame["previous_invoice_count"] == 0, history_features].isna().all().all(), (
    "Missing first-history values must remain NaN for train-fitted imputation."
)
print("Forbidden leakage features: NONE")
print("Feature count, split disjointness, temporal ordering, and history missingness: PASS")

## 12. Feature Dictionary

In [ ]:
feature_dictionary = pd.DataFrame([
    ["invoice_amount", "float", "raw invoice","invoice_amount","Invoice amount known before payment", True, True, float(feature_frame["invoice_amount"].isna().mean()*100)],
    ["log_invoice_amount", "float", "Feature-engine LogTransformer", "log(invoice_amount)", "Stabilized invoice magnitude for skewed spending", True, True, float(feature_frame["log_invoice_amount"].isna().mean()*100)],
    ["invoice_year", "int", "Feature-engine DatetimeFeatures", "invoice_date.year", "Calendar year of invoice", True, True, float(feature_frame["invoice_year"].isna().mean()*100)],
    ["invoice_month", "int", "Feature-engine DatetimeFeatures", "invoice_date.month", "Calendar month of invoice", True, True, float(feature_frame["invoice_month"].isna().mean()*100)],
    ["invoice_quarter", "int", "Feature-engine DatetimeFeatures", "invoice_date.quarter", "Quarter of invoice", True, True, float(feature_frame["invoice_quarter"].isna().mean()*100)],
    ["invoice_day_of_week", "int", "Feature-engine DatetimeFeatures", "invoice_date.day_of_week", "Weekday signal", True, True, float(feature_frame["invoice_day_of_week"].isna().mean()*100)],
    ["invoice_day_of_month", "int", "Feature-engine DatetimeFeatures", "invoice_date.day", "Day of month", True, True, float(feature_frame["invoice_day_of_month"].isna().mean()*100)],
    ["invoice_week_of_year", "int", "Feature-engine DatetimeFeatures", "invoice_date.week_of_year", "Week-of-year", True, True, float(feature_frame["invoice_week_of_year"].isna().mean()*100)],
    ["invoice_is_weekend", "bool", "Feature-engine DatetimeFeatures", "invoice_date.is_weekend", "Weekend indicator", True, True, float(feature_frame["invoice_is_weekend"].isna().mean()*100)],
    ["days_until_due", "int", "derived rule", "(due_date - invoice_date).dt.days", "Lead time to invoice due date", True, True, float(feature_frame["days_until_due"].isna().mean()*100)],
    ["previous_invoice_count", "int", "custom LeakageSafeCustomerHistoryTransformer", "count(previous invoices)", "Number of earlier invoices for the same customer", True, True, float(feature_frame["previous_invoice_count"].isna().mean()*100)],
    ["previous_late_count", "float", "custom LeakageSafeCustomerHistoryTransformer", "sum(previous is_late)", "Earlier late invoices for same customer", True, True, float(feature_frame["previous_late_count"].isna().mean()*100)],
    ["previous_late_ratio", "float", "custom LeakageSafeCustomerHistoryTransformer", "previous_late_count / previous_invoice_count", "Long-run late-rate signal", True, True, float(feature_frame["previous_late_ratio"].isna().mean()*100)],
    ["previous_avg_delay", "float", "custom LeakageSafeCustomerHistoryTransformer", "mean(previous delay_days)", "Average prior delay for the customer", True, True, float(feature_frame["previous_avg_delay"].isna().mean()*100)],
    ["previous_median_delay", "float", "custom LeakageSafeCustomerHistoryTransformer", "median(previous delay_days)", "Median prior delay", True, True, float(feature_frame["previous_median_delay"].isna().mean()*100)],
    ["previous_avg_invoice_amount", "float", "custom LeakageSafeCustomerHistoryTransformer", "mean(previous invoice_amount)", "Average historical invoice size", True, True, float(feature_frame["previous_avg_invoice_amount"].isna().mean()*100)],
    ["previous_total_invoice_amount", "float", "custom LeakageSafeCustomerHistoryTransformer", "sum(previous invoice_amount)", "Historical total spend for customer", True, True, float(feature_frame["previous_total_invoice_amount"].isna().mean()*100)],
    ["days_since_previous_invoice", "float", "custom LeakageSafeCustomerHistoryTransformer", "(current_date - last_prior_date).days", "Recency of the most recent prior invoice", True, True, float(feature_frame["days_since_previous_invoice"].isna().mean()*100)],
    ["has_payment_history", "int", "custom LeakageSafeCustomerHistoryTransformer", "1 if earlier invoices exist else 0", "Indicator of prior-customer payment history", True, True, float(feature_frame["has_payment_history"].isna().mean()*100)],
], columns=["feature_name", "data_type", "source", "calculation", "description", "available_at_prediction_time", "leakage_safe", "missing_percentage"])
report_dir.mkdir(parents=True, exist_ok=True)
feature_dictionary.to_csv(report_path, index=False)
print("Feature dictionary saved to:", report_path)
print(feature_dictionary.head(5).to_string(index=False))


## 13. Save Outputs

In [ ]:
feature_dataset = feature_frame[["invoice_id", "customer_id", "invoice_date", "is_late"] + final_feature_cols].copy()
feature_dataset.to_csv(feature_output_path, index=False)
train_split = train[["invoice_date", "invoice_id", "customer_id", "is_late"] + final_feature_cols].copy()
validation_split = validation[["invoice_date", "invoice_id", "customer_id", "is_late"] + final_feature_cols].copy()
test_split = test[["invoice_date", "invoice_id", "customer_id", "is_late"] + final_feature_cols].copy()
train_split.to_csv(train_output_path, index=False)
validation_split.to_csv(validation_output_path, index=False)
test_split.to_csv(test_output_path, index=False)
joblib.dump({
    "history_transformer": history_transformer,
    "feature_pipeline": feature_pipeline,
}, pipeline_output_path)
print("Saved feature dataset:", feature_output_path)
print("Saved train split:", train_output_path)
print("Saved validation split:", validation_output_path)
print("Saved test split:", test_output_path)
print("Saved pipeline:", pipeline_output_path)


## 13.5. Historical Leakage Proof

A compact counterfactual test verifies that same-day invoices do not count as prior history and that changing a later invoice's outcome cannot alter earlier features.

In [ ]:
leakage_probe = pd.DataFrame({
    "invoice_id": [1, 2, 3, 4],
    "customer_id": ["probe", "probe", "probe", "probe"],
    "invoice_date": pd.to_datetime(["2024-01-01", "2024-01-01", "2024-01-02", "2024-01-03"]),
    "is_late": [1, 0, 1, 0],
    "invoice_amount": [100.0, 200.0, 300.0, 400.0],
    "delay_days": [10.0, 0.0, 5.0, 0.0],
})
probe_transformer = LeakageSafeCustomerHistoryTransformer()
probe_history = probe_transformer.fit_transform(leakage_probe)

assert probe_history.loc[0, "previous_invoice_count"] == 0
assert probe_history.loc[1, "previous_invoice_count"] == 0
assert probe_history.loc[2, "previous_invoice_count"] == 2
assert probe_history.loc[2, "previous_late_count"] == 1
assert probe_history.loc[2, "previous_late_ratio"] == 0.5
assert probe_history.loc[3, "previous_invoice_count"] == 3

perturbed_probe = leakage_probe.copy()
perturbed_probe.loc[3, ["is_late", "invoice_amount", "delay_days"]] = [1, 9999.0, 999.0]
perturbed_history = probe_transformer.transform(perturbed_probe)
np.testing.assert_allclose(
    probe_history.loc[:2].to_numpy(dtype=float),
    perturbed_history.loc[:2].to_numpy(dtype=float),
    equal_nan=True,
)

assert len(final_feature_cols) == 19
assert set(final_feature_cols).isdisjoint(FORBIDDEN_FEATURES)
print("Strict-date and future-perturbation leakage checks passed.")

## 14. Final Validation Report

In [ ]:
reloaded_artifacts = joblib.load(pipeline_output_path)
assert set(reloaded_artifacts) == {"history_transformer", "feature_pipeline"}
reloaded_train_trans = reloaded_artifacts["feature_pipeline"].transform(train[final_feature_cols])
np.testing.assert_allclose(train_trans, reloaded_train_trans, rtol=1e-12, atol=1e-12)
reloaded_probe_history = reloaded_artifacts["history_transformer"].transform(leakage_probe)
np.testing.assert_allclose(
    probe_history.to_numpy(dtype=float),
    reloaded_probe_history.to_numpy(dtype=float),
    equal_nan=True,
)

print("FLOWCRED AI")
print("LATE-PAYMENT FEATURE ENGINEERING")
print("Paid invoices:", len(paid_df))
print("Customers:", paid_df["customer_id"].nunique(dropna=True))
print("Training rows:", len(train))
print("Validation rows:", len(validation))
print("Test rows:", len(test))
print("Number of features:", len(final_feature_cols))
print("Feature-engine transformations: [DatetimeFeatures, LogTransformer]")
print("Custom leakage-safe transformations: [LeakageSafeCustomerHistoryTransformer]")
print("Forbidden leakage features: NONE")
print("Temporal split and disjointness: PASS")
print("Strict prior-date and future-perturbation tests: PASS")
print("Pipeline serialization/reload equivalence: PASS")
print("Feature pipeline: SAVED")
print("Feature dataset: SAVED")
